In [69]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker

In [70]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_embryo2/")

Prepare data

In [71]:
rna = sc.read_h5ad("rna.h5ad")
atac = sc.read_h5ad("atac.h5ad")

In [72]:
(rna.obs_names == atac.obs_names).all()

np.True_

In [73]:
atac.obsm["spatial"] = rna.obsm["spatial"].copy()

In [74]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))

        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [75]:
# h5ad_to_h5(rna, output_file="rna.h5")
# h5ad_to_h5(atac, output_file="atac.h5")

In [76]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [77]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_embryo2/"
# bm.run(methods=["Seurat_WNN"
#                ],
#        RNA_file_path=data_folder+"rna.h5",
#        ATAC_file_path=data_folder+"/atac.h5",
#        n_cluster=7,
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_embryo2/")

In [78]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_embryo2/smopca.csv", index_col=0)

In [79]:
# if "cluster" in rna.obs: del rna.obs["cluster"]
# if "cluster_colors" in rna.uns: del rna.uns["cluster_colors"]
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# rna.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# rna.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [80]:
# sc.pl.umap(rna, color="cluster")
# sc.pl.spatial(rna, color="cluster", spot_size=1)

In [81]:
# methods =  ["Seurat_WNN",    "MOFA2",    "MultiVI",  "Multigrate",    "scMM",       "scMDC",
#                 "Matilda",   "moETM",  "MISO",   "SpatialGlue",  "COSMOS",     "PRESENT",
#                 "SMOPCA",       "CellCharter"
#                ]
# for m in methods:
#     res = pd.read_csv(f"/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_embryo2/{m.lower()}.csv", index_col=0)
#     res.columns = ["UMAP1", "UMAP2", "cluster"]
#     cluster = len(set(res["cluster"]))
#     print(m, cluster)

Plot

In [82]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_embryo2/"
methods = ["Seurat_WNN", "MOFA2", "MultiVI", "Multigrate", "scMM", "scMDC", "Matilda", "moETM",
"MISO", "SpatialGlue",  "COSMOS", "PRESENT", "SMOPCA", "CellCharter"]
res = bm.read_result(path=result_folder,
                     methods=methods + ["rna", "atac"],
                     reindex=False)

2026-04-01 21:54:53 - WARNING - '_latent' result for 'Seurat_WNN' not found at: /mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Mouse_embryo2/seurat_wnn_latent.csv


In [83]:
from benchmarker import  read_sparse_h5, recompute_aggregate_scores

In [84]:
res["Embed"]["Seurat_WNN"] = res["Embed"]["SpatialGlue"].copy()
res["seurat_wnn_conn"] = read_sparse_h5(f"{result_folder}/seurat_wnn_connection.h5")[0]
res["seurat_wnn_dist"] = read_sparse_h5(f"{result_folder}/seurat_wnn_distance.h5")[0]

In [85]:
rna.obs["batch"] = ["batch1"]*1000 + ["batch2"] * (rna.shape[0]-1000)

In [86]:
rna.obs["cell_type"] = atac.obs["annot"].copy()

In [87]:
# metrics = bm.cal_metrics(adata=rna, batch_key="batch", label_key="cell_type",
#                          res_dict=res, methods="all", verbose=True, rep=1,
#                          min_max_scale=False,
#                          save=f"{result_folder}/metrics.pkl")

In [88]:
with open(f"{result_folder}/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)

In [89]:
metric = metrics[0][['Isolated labels', 'NMI', 'ARI', 'Silhouette label',
       'cLISI', 'CHAOS', 'PAS', 'Domain continuity','Bio conservation']]

In [90]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_paired/Mouse_embryo2"

In [91]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [92]:
metric = recompute_aggregate_scores(metric)
metric = metric.drop(["rna", "atac"])

In [93]:
metric.columns = ['Isolated labels', 'NMI', 'ARI', 'Silhouette label', 'cLISI', 'CHAOS',
       'PAS', 'Domain continuity', 'Bio conservation']

In [95]:
# bm.plot_heatmap(metric_df=metric, total_name="Bio conservation",
#                 save=f"{figure_save_dir}/summary_heatmap_all.pdf",
#                 # show_top=7,
#                 # show_bottom=0,
#                 # insert_marker_row = 8,
#                 )

In [54]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [rna.obsm["spatial"]]
spatial = transform_coord(spatial, vertical=False, horizontal=False, angle=90)

In [58]:
spatial_methods = ["COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#5873a4"
bg_dict["ATAC"] = "#5873a4"
bg_dict["Protein"] = "#5873a4"
bg_dict["Annotation"] = "#97a4af"

In [65]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(1.97, 4.5),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=1,
#                 xlabel=["RNA", "ATAC"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "atac"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_single_modal.pdf",
#                 rasterized=True
#                 )

In [76]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(12, 4.5),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=6,
#                 xlabel=["MultiVI", "Seurat_WNN", "Matilda", "MOFA2","scMDC", "scMM",
#                 "COSMOS", "CellCharter", "SMOPCA", "SpatialGlue", "PRESENT", "MISO"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["MultiVI", "Seurat_WNN", "Matilda", "MOFA2","scMDC", "scMM",
#                 "COSMOS", "CellCharter", "SMOPCA", "SpatialGlue", "PRESENT", "MISO"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods.pdf",
#                 rasterized=True
#                 )

In [65]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(12, 6.8),
#                 frameon=True,
#                 inner_gs_row=1,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=6,
#                 xlabel=["RNA", "ATAC","Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["rna", "atac","Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#                 outer_row_hspace=0.22,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 # inner_col_wspace = -0.12,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.012, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods_all.pdf",
#                 rasterized=True
#                 )

In [60]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict={"annot": np.array(rna.obs["cell_type"]).reshape(-1,1)},
#                 figsize=(1.97, 2.03),
#                 frameon=True,
#                 inner_gs_row=1, inner_gs_col=1,
#                 size=20,
#                 ncol=1,
#                 xlabel=["Annotation"],
#                 ylabel=None,
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 outer_row_hspace=0.15,
#                 outer_col_wspace=0.1,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.015,
#                 save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_annot.pdf"
#                 )

In [68]:
# bm.plot_legend(category_lst=rna.obs["cell_type"],
#                 marker="o",
#                 ncol=1,
#                 save=f"{figure_save_dir}/annot_legend.pdf",
#                 # order=["CA3 Pyr", "Chroid plexus", "GCL", "White matter", "Hilus", "ML",  "SLM"]
#                 )

In [74]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=None,
#              annot_list=list(rna.obs["cell_type"]),
#              figsize=(10, 6.7),
#              frameon=True,
#              inner_gs_row=1,
#              inner_gs_col=1,
#              size=15,
#              ncol=5,
#              xlabel=["RNA", "ATAC", "MultiVI", "Seurat_WNN", "Matilda", "MOFA2","scMDC", "scMM",
#                 "COSMOS", "CellCharter", "SMOPCA", "SpatialGlue", "PRESENT", "MISO"],
#              only_show_top=False,
#              ylabel=None,
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["rna", "atac", "MultiVI", "Seurat_WNN", "Matilda", "MOFA2","scMDC", "scMM",
#                 "COSMOS", "CellCharter", "SMOPCA", "SpatialGlue", "PRESENT", "MISO"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#             #  ylabel_pad=0.02,
#              xlabel_pad=0.014,
#              outer_row_hspace=0.22,
#              merge=False,
#              merge_margin_size=0.4,
#              save=f"{figure_save_dir}/umap_methods.pdf"
# )

In [66]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=None,
#              annot_list=list(rna.obs["cell_type"]),
#              figsize=(16, 4.5),
#              frameon=True,
#              inner_gs_row=1,
#              inner_gs_col=1,
#              size=15,
#              ncol=8,
#              xlabel=["RNA", "ATAC","Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#              only_show_top=False,
#              ylabel=None,
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["rna","atac", "Matilda", "MultiVI", "Seurat_WNN", "scMDC", "scMM", "moETM", "Multigrate", "MOFA2",
#                 "COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.02,
#              save_dpi=600,
#             #  ylabel_pad=0.02,
#              xlabel_pad=0.012,
#              outer_row_hspace=0.22,
#              merge=False,
#              merge_margin_size=0.4,
#              save=f"{figure_save_dir}/umap_methods_all.pdf"
# )